# Hands-on 2: is tonight's data still like the training data?
AICTE QIP · Artificial Intelligence and Engineering · IIIT Kottayam

Tonight the class sent real requests to the live app. Here you check them the way a monitoring system would.

Run each cell with **Shift + Enter**. Change only the lines marked `# FILL`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

train = pd.read_csv("https://raw.githubusercontent.com/ansiths-iiitk/ews-demo/main/training_data.csv")

USE_UPLOAD = False   # set to True to upload the tonight.csv your instructor shares
if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    tonight = pd.read_csv(next(iter(uploaded)))
else:
    tonight = pd.read_csv("https://raw.githubusercontent.com/ansiths-iiitk/ews-demo/main/sample_tonight.csv")   # a prepared class session

print(len(tonight), "requests in the log")
tonight[["ts", "version", "attendance_raw", "marks_raw", "status", "label"]].head(8)

## Step 1 · Keep only requests that make sense
Some requests crashed, some were rejected, and in release v1.0 some impossible values (like marks of 1000) were accepted. A monitoring check should not be fooled by those.

In [ ]:
ok = tonight[tonight["status"] == "ok"]
ok = ok[ok["marks"].between(0, 100) & ok["attendance"].between(___, 100)].reset_index(drop=True)   # FILL
print(len(ok), "sensible requests kept")

## Step 2 · Compare the averages

In [ ]:
print("Average mark in training data:", round(train["marks"].mean(), 1))
print("Average mark tonight:         ", round(ok["marks"].___(), 1))      # FILL

## Step 3 · Look at the whole picture, not just the average

In [ ]:
bins = range(0, 101, 10)
plt.figure(figsize=(8, 3.8))
plt.hist(train["marks"], bins=bins, weights=np.ones(len(train)) / len(train),
         histtype="step", linewidth=2, label="Training data")
plt.hist(ok["marks"], bins=bins, weights=np.ones(len(ok)) / len(ok),
         alpha=0.45, label="Tonight")
plt.xlabel("Internal marks"); plt.ylabel("Share of students"); plt.legend()
plt.title("Marks: training data vs tonight"); plt.show()

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.scatter(range(len(ok)), ok["marks"], s=14)
plt.xlabel("Request number (in the order they arrived)"); plt.ylabel("Marks")
plt.title("Marks of each request, in arrival order"); plt.show()

**Discuss:** at roughly which request does the pattern change? What happened in the class at that moment?

*Your answer:*

## Step 4 · An automatic drift alarm
Real systems cannot rely on someone staring at a chart. Here is a very simple rule: raise an alarm when the average mark in a window of recent requests is far from the training average.

In [ ]:
TRAIN_MEAN = train["marks"].mean()

def drift_alarm(window, threshold=15):
    gap = abs(window["marks"].mean() - TRAIN_MEAN)
    return gap > ___                      # FILL: compare with the threshold

size = 20
for start in range(0, len(ok) - size + 1, 10):
    w = ok.iloc[start:start + size]
    print(f"requests {start:3d}-{start + size - 1:3d}   average mark {w['marks'].mean():5.1f}   "
          f"alarm: {'YES' if drift_alarm(w) else 'no'}")

## Step 5 · Which release did what?
During the evening the app ran **v1.0**, then **v1.1**, a rushed **v2.0**, a rollback to **v1.1**, and finally **v2.1**. Compare them: what went in, and what came out.

In [ ]:
summary = ok.groupby(___).agg(                          # FILL: the column that names the release
    requests=("label", "size"),
    avg_attendance=("attendance", "mean"),
    avg_marks=("marks", "mean"),
    flagged_share=("label", lambda s: (s == "Needs review").mean()),
).round(2)
summary

In [ ]:
# the same flagged share, request by request, with the release that answered each one
ok["flagged"] = (ok["label"] == "Needs review").astype(int)
ok["recent"] = ok["flagged"].rolling(10, min_periods=5).mean() * 100
plt.figure(figsize=(9, 3.5))
plt.plot(ok.index, ok["recent"], color="black")
for v, g in ok.groupby((ok["version"] != ok["version"].shift()).cumsum()):
    plt.axvspan(g.index[0] - 0.5, g.index[-1] + 0.5, alpha=0.08 if g["version"].iloc[0] != "v2.0" else 0.25)
    plt.text(g.index[0], 103, g["version"].iloc[0], fontsize=9)
plt.ylim(0, 110); plt.ylabel("% flagged (last 10)"); plt.xlabel("Request number")
plt.title("Flagged share through the evening"); plt.show()

**Discuss:**
1. Which release made the answers worse? How can you tell it was the release and not the students?
2. Did v2.1 bring the marks back to the scale the model learned? Look at `avg_marks`.
3. You are on call when v2.0 goes live. What do you do first: retrain the model, fix the code, or roll back?
4. v2.1 asks the user to choose "out of 20" or "out of 100". What can still go wrong?

*Your answers:*